# Repository Selection

Construction of the candidate cohort for the benchmark. The workflow profiles repository metadata, applies structural constraints, scores review evidence under finite sample uncertainty, and fixes the cohort at the chosen threshold.

In [1]:
from datetime import datetime, timezone

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from lingua import LanguageDetectorBuilder
from plotly.subplots import make_subplots
from scipy.stats import entropy
from statsmodels.stats.proportion import proportion_confint

from benchmark_notebook.io import CloudStorage

pio.templates["thesis"] = go.layout.Template(
    layout=go.Layout(
        font=dict(
            family="TeX Gyre Heros, Helvetica, Arial, sans-serif",
            size=11,
            color="#1A1A1A",
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        bargap=0.02,
        colorway=[
            # "#0072B2",
            # "#009E73",
            # "#E69F00",
            # "#CC79A7",
            # "#56B4E9",
            # "#D55E00",
            # "#F0E442",
            # "#000000",
            "#1A1A1A",
            "#808080",
            "#BFBFBF",
            "#404040",
            "#666666",
        ],
        colorscale=dict(
            sequential="Viridis",
            diverging="RdBu",
        ),
        xaxis=dict(
            title=dict(standoff=20, font=dict(size=12)),
            showline=False,
            zeroline=False,
            showgrid=True,
            gridcolor="#E8E8E8",
            gridwidth=0.5,
        ),
        yaxis=dict(
            title=dict(standoff=20, font=dict(size=12)),
            showline=False,
            zeroline=False,
            showgrid=True,
            gridcolor="#E8E8E8",
            gridwidth=0.5,
        ),
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.16,
            xanchor="right",
            x=1.03,
            font=dict(size=12),
        ),
        margin=dict(l=80, r=30, t=2, b=2, pad=10),
    ),
    data=dict(
        scatter=[
            go.Scatter(
                line=dict(
                    width=1,
                )
            )
        ]
    ),
)

pio.templates.default = "thesis"

In [3]:
from pathlib import Path

cache = Path("/usr/local/src/.cache")
cache.mkdir(parents=True, exist_ok=True)

## Distributions

The candidate population is described along the dimensions that govern subsequent selection. Age, target language volume, target language share, merge activity, and English confidence motivate the thresholds applied in the following section. Their empirical shape grounds the choice of cutoffs in observation rather than in prior assumption.

In [5]:
now = datetime(2025, 9, 1, tzinfo=timezone.utc)
path = cache / "meta.parquet"

if path.exists():
    meta = pl.read_parquet(path)
else:
    meta = pl.from_arrow(
        CloudStorage(bucket="bug-request-bucket").asset(
            "scope-repository-profile",
            location="main",
            partition="default",
            version="2.0.0",
        )
    )

    detector = (
        LanguageDetectorBuilder.from_all_languages()
        .with_preloaded_language_models()
        .build()
    )

    def detect(excerpts):
        bodies = [e["body"] for e in excerpts if e and e.get("body")]
        if not bodies:
            return []
        blob = " ".join(bodies)
        return [
            {
                "language": c.language.iso_code_639_1.name.lower(),
                "confidence": float(c.value),
            }
            for c in detector.compute_language_confidence_values(blob)
            if c.value > 0.0
        ]

    targets = {"Python"}

    excluded = {
        "C",
        "C++",
        "C#",
        "Java",
        "Kotlin",
        "Objective-C",
        "Swift",
        "Rust",
        "Go",
        "Scala",
        "Dart",
        "PHP",
        "Ruby",
        "JavaScript",
        "TypeScript",
        "CoffeeScript",
        "Groovy",
        "Clojure",
        "F#",
        "Visual Basic",
        "Visual Basic .NET",
        "Zig",
        "D",
        "Nim",
        "Crystal",
        "V",
        "Ada",
        "Fortran",
        "Assembly",
        "Haskell",
        "OCaml",
        "Erlang",
        "Elixir",
        "Elm",
        "Lisp",
        "Common Lisp",
        "Scheme",
        "Racket",
        "PureScript",
        "ReScript",
        "Haxe",
        "Perl",
        "Lua",
        "R",
        "Julia",
        "Tcl",
        "Objective-C++",
        "MATLAB",
        "Mathematica",
        "Solidity",
    }

    meta = meta.with_columns(
        pl.col("excerpts")
        .map_elements(
            detect,
            return_dtype=pl.List(
                pl.Struct({"language": pl.Utf8, "confidence": pl.Float64})
            ),
        )
        .alias("tongue"),
        pl.col("languages")
        .list.eval(
            pl.when(pl.element().struct.field("name").is_in(targets))
            .then(pl.element().struct.field("size"))
            .otherwise(0)
        )
        .list.sum()
        .alias("target_language_size"),
        pl.col("languages")
        .list.eval(
            pl.when(pl.element().struct.field("name").is_in(excluded))
            .then(pl.element().struct.field("size"))
            .otherwise(0)
        )
        .list.sum()
        .alias("excluded_language_size"),
    ).with_columns(
        (pl.col("target_language_size") / pl.col("size").clip(lower_bound=1)).alias(
            "target_language_share"
        ),
        (pl.col("excluded_language_size") / pl.col("size").clip(lower_bound=1)).alias(
            "excluded_language_share"
        ),
        (
            (
                pl.lit(now)
                - pl.col("creation").str.to_datetime(
                    format="%Y-%m-%dT%H:%M:%SZ", time_zone="UTC"
                )
            ).dt.total_days()
            / 365
        ).alias("age_years"),
    )
    meta.write_parquet(path)

In [20]:
plot = meta.with_columns(
    pl.when(pl.col("merges") > 0)
    .then(pl.col("merges").log10())
    .otherwise(None)
    .alias("log10_merges"),
    pl.col("target_language_size").log10().alias("log10_target_language_size"),
    pl.when(
        pl.col("tongue").list.get(0, null_on_oob=True).struct.field("language") == "en"
    )
    .then(pl.col("tongue").list.get(0, null_on_oob=True).struct.field("confidence"))
    .otherwise(None)
    .alias("english_confidence"),
)

panels = [
    {"x": "age_years", "xaxis": "Age in Years"},
    {"x": "log10_target_language_size", "xaxis": "Python Volume on Log Scale"},
    {"x": "target_language_share", "xaxis": "Python Share"},
    {"x": "log10_merges", "xaxis": "Merges on Log Scale"},
    {"x": "english_confidence", "xaxis": "English Confidence"},
]

bins = 50

fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
)

for i, panel in enumerate(panels):
    row, col = divmod(i, 2)
    row, col = row + 1, col + 1

    series = plot[panel["x"]].drop_nulls()
    finite = series.filter(series.is_finite())
    lo, hi = float(finite.min()), float(finite.max())

    fig.add_trace(
        go.Histogram(
            x=plot[panel["x"]],
            marker=dict(
                color=pio.templates["thesis"].layout.colorway[4],
                line=dict(width=0),
            ),
            xbins=dict(start=lo, end=hi + (hi - lo) / bins, size=(hi - lo) / bins),
            autobinx=False,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(title_text=panel["xaxis"], row=row, col=col)
    fig.update_yaxes(title_text="Repositories", row=row, col=col)

fig.update_layout(
    height=800,
    width=800,
    showlegend=False,
    bargap=0.02,
    margin=dict(l=70, r=30, t=50, b=60),
).show()

## Constraints

Structural prerequisites are imposed to remove repositories that are incompatible with the intended analysis scope. The constraints concern primary language, language composition, age, project size, merge activity, and English communication. Each constraint reflects a necessary condition for the noise model rather than a downstream optimization target.

In [7]:
filtered = meta.filter(
    (pl.col("age_years") >= 1)
    & (pl.col("size") >= 300_000)
    & (pl.col("target_language_share") >= 0.90)
    & (pl.col("excluded_language_share") < 0.10)
    & (pl.col("language").is_in(targets))
    & (pl.col("merges") >= 100)
    & (pl.col("archive") == False)
    & (
        pl.col("tongue")
        .list.eval(
            pl.when(pl.element().struct.field("language") == "en")
            .then(pl.element().struct.field("confidence"))
            .otherwise(0)
        )
        .list.sum()
        >= 0.95
    )
    & (
        pl.col("tongue")
        .list.eval(
            pl.when(pl.element().struct.field("language") != "en")
            .then(pl.element().struct.field("confidence"))
            .otherwise(0)
        )
        .list.sum()
        < 0.05
    )
)

## Scoring

The empirical review proportion is unstable in repositories with few merges. Wilson score intervals and Bayesian shrinkage are introduced as finite sample evidence proxies. The asymmetric Wilson score rewards repositories whose interval reaches into the strict review regime. The cost curve compares the resulting populations under alternative scoring functions across threshold values.

In [47]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import norm

tau = 0.75
palette = pio.templates["thesis"].layout.colorway

methods = [
    ("review_share", "Raw", palette[4], "dash"),
    ("wilson_asymmetric", "Asymmetric Wilson", palette[4], "solid"),
]

merges = candidates["total_merges"]
grid = np.unique(np.round(np.logspace(0, np.log10(merges.max()), 400)).astype(int))
z = norm.ppf(0.975)
shares = np.linspace(0.001, 0.999, 10000)

lower = np.empty_like(grid, dtype=float)
asymmetric = np.empty_like(grid, dtype=float)
for i, n in enumerate(grid):
    denom = 1 + z**2 / n
    centre = (shares + z**2 / (2 * n)) / denom
    half = (z / denom) * np.sqrt(shares * (1 - shares) / n + z**2 / (4 * n**2))
    lo = np.clip(centre - half, 0, 1)
    hi = np.clip(centre + half, 0, 1)
    lower[i] = shares[np.argmin(np.abs(lo - tau))]
    asymmetric[i] = shares[np.argmin(np.abs((hi - (hi - lo) * (1 - hi)) - tau))]

isos = {
    "review_share": np.full_like(grid, tau, dtype=float),
    "wilson_asymmetric": asymmetric,
}

thresholds = np.linspace(0.5, 1.0, 101)
costs = {
    column: np.array(
        [
            candidates.filter(pl.col(column) >= t)["total_changes"].sum()
            for t in thresholds
        ]
    )
    for column, *_ in methods
}

cloud = go.Scatter(
    x=merges,
    y=candidates["review_share"],
    mode="markers",
    marker=dict(size=3, color=pio.templates["thesis"].layout.colorway[2]),
    showlegend=False,
)

curves = [
    go.Scatter(
        x=grid,
        y=isos[column],
        mode="lines",
        line=dict(color=color, dash=dash, width=1),
        name=label,
        legendgroup=label,
    )
    for column, label, color, dash in methods
]

costlines = [
    go.Scatter(
        x=thresholds,
        y=costs[column],
        mode="lines",
        line=dict(color=color, dash=dash, width=1),
        name=label,
        legendgroup=label,
        showlegend=False,
    )
    for column, label, color, dash in methods
]

(
    make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.12,
    )
    .add_traces([cloud, *curves], rows=1, cols=1)
    .add_traces(costlines, rows=1, cols=2)
    .update_xaxes(title_text="Total Merges", type="log", dtick=1, row=1, col=1)
    .update_yaxes(title_text="Review Share", range=[0, 1.02], row=1, col=1)
    .update_xaxes(title_text="Threshold", row=1, col=2)
    .update_yaxes(title_text="Pull Requests", row=1, col=2)
    .update_layout(width=1000, height=450, margin=dict(l=70, r=30, t=40, b=60))
    .show(config={"toImageButtonOptions": {"format": "svg"}})
)

## Cohort

The metadata filter and the asymmetric Wilson score are combined at the threshold fixed by the prestudy analysis. The resulting repositories form the candidate cohort that enters integration branch reconstruction and review discipline profiling. The printed name list documents the cohort for reproducibility.

In [9]:
threshold = 0.75

cohort = (
    filtered.join(candidates, left_on="name", right_on="name", how="inner")
    .filter(pl.col("wilson_asymmetric") >= threshold)
    .unique(subset=["name"])
    .sort("name")
)

with pl.Config(
    tbl_rows=-1,
    tbl_cols=-1,
    tbl_width_chars=-1,
    fmt_str_lengths=200,
):
    print(cohort["name"])

shape: (635,)
Series: 'name' [str]
[
	"309thEDDGE/weave"
	"AMReX-Codes/pyamrex"
	"AMWA-TV/nmos-testing"
	"ARISE-Initiative/robosuite"
	"ARM-DOE/ACT"
	"AequilibraE/qaequilibrae"
	"AlibabaPAI/llumnix"
	"AllenNeuralDynamics/aind-data-schema"
	"AllenNeuralDynamics/aind-data-schema-models"
	"AnkiHubSoftware/ankihub_addon"
	"ApeWorX/ape"
	"ArcadeAI/arcade-ai"
	"ArchipelagoMW/Archipelago"
	"Azure/WALinuxAgent"
	"Azure/azure-cli"
	"Azure/azure-cli-extensions"
	"Azure/azure-functions-durable-python"
	"Azure/azure-functions-python-worker"
	"Azure/azure-sdk-for-python"
	"AzureAD/microsoft-authentication-library-for-python"
	"Ballsdex-Team/BallsDex-DiscordBot"
	"BerkeleyLearnVerify/Scenic"
	"Blueprints-org/blueprints"
	"Bogdanp/dramatiq"
	"BoostryJP/ibet-Prime"
	"BuildACell/bioCRNpyler"
	"CAREamics/careamics"
	"CERTCC/SSVC"
	"CITCOM-project/CausalTestingFramework"
	"CSSUoB/TeX-Bot-Py-V2"
	"CUQI-DTU/CUQIpy"
	"CWorthy-ocean/C-Star"
	"CadQuery/cadquery"
	"ChipFlow/chipflow-lib"
	"CiscoDevNet/ansible-